In [1]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

In [2]:
from tempfile import TemporaryDirectory
from PIL import Image
from klygo import datasets, files, media

if 'tutorial_workspace' in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix='klygo-datasets-convert-')
root = files.path(tutorial_workspace.name)
print('Initialized temporary tutorial workspace:', root)

Initialized temporary tutorial workspace: C:\Users\Mine\AppData\Local\Temp\klygo-datasets-convert-9b412_wz


# Convert raw dataset trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Convert.ipynb)

## Jump to an API

- [datasets.convert](#datasets.convert)

### datasets.convert

`datasets.convert(source, target, format='yolo', tree=None, classes=None, on_conflict='error', overwrite=False) -> pathlib.Path`

- `tree`: ordered image, label, and optional YAML source patterns.
- `classes`: contiguous class mapping when no YAML is supplied.
- `on_conflict`: `error`, `rename`, `skip`, or `overwrite`.
- Returns the atomically published flat dataset directory.

#### Basic use

In [3]:
source = files.join(root, 'raw')
for batch, stem in (('a', 'first'), ('b', 'second')):
    images = files.join(source, batch, 'images')
    labels = files.join(source, batch, 'labels')
    files.mkdir(images)
    files.mkdir(labels)
    media.save(files.join(images, stem + '.jpg'), Image.new('RGB', (64, 48), 'white'), verbose=False)
    files.save(files.join(labels, stem + '.txt'), ['0 0.5 0.5 0.4 0.4'], verbose=False)
flat = datasets.convert(source, files.join(root, 'flat'), tree={'images': '*/images', 'labels': '*/labels'}, classes={0: 'item'})
print(datasets.info(flat, verbose=False))

{'classes': ['item'], 'class_ids': [0], 'nc': 1, 'total_images': 2, 'total_labels': 2, 'unlabeled_images': 0, 'splits': {'raw': 2}}


#### Practical variation

Use `on_conflict='rename'` when different batches contain the same basename.

In [4]:
dup_source = files.join(root, 'raw_dup')
for batch in ('batch1', 'batch2'):
    img_dir = files.join(dup_source, batch, 'images')
    lbl_dir = files.join(dup_source, batch, 'labels')
    files.mkdir(img_dir)
    files.mkdir(lbl_dir)
    media.save(files.join(img_dir, 'frame.jpg'), Image.new('RGB', (64, 48), 'gray'), verbose=False)
    files.save(files.join(lbl_dir, 'frame.txt'), ['0 0.5 0.5 0.3 0.3'], verbose=False)
renamed_flat = datasets.convert(
    dup_source,
    files.join(root, 'flat_renamed'),
    tree={'images': '*/images', 'labels': '*/labels'},
    classes={0: 'item'},
    on_conflict='rename',
)
print('Renamed images count:', datasets.info(renamed_flat, verbose=False)['total_images'])

Renamed images count: 2


#### Expected error and recovery

An existing target raises `FileExistsError`; pass `overwrite=True` only when replacing the complete target is intended.

In [5]:
try:
    datasets.convert(source, flat, tree={'images': '*/images', 'labels': '*/labels'}, classes={0: 'item'})
except FileExistsError as exc:
    print('Caught expected FileExistsError:', exc)

Caught expected FileExistsError: target already exists: C:\Users\Mine\AppData\Local\Temp\klygo-datasets-convert-9b412_wz\flat


## Put it together

Convert raw sources, then call `datasets.partition(flat, partitioned, (0.5, 0.5, 0.0))` to create train and validation splits.

In [6]:
partitioned = datasets.partition(flat, files.join(root, 'partitioned'), ratios=(0.5, 0.5, 0.0), overwrite=True, verbose=False)
print('Partitioned splits:', datasets.info(partitioned, verbose=False)['splits'])

Partitioned splits: {'train': 1, 'val': 1}


## Remember

Tree roles follow insertion order, conflicts apply to complete image-label samples, and conversion never remaps class meaning. Return to [the datasets overview](Document_Datasets_Overview.ipynb) or continue with [partition](Document_Datasets_Partition.ipynb).